In [ ]:
import streamlit as st
import cv2
from ultralytics import YOLO


model = YOLO("yolov8s.pt")


cap = cv2.VideoCapture("foot ball.mp4")

if not cap.isOpened():
    print("Error: Cannot open video file.")
    exit()


fps = cap.get(cv2.CAP_PROP_FPS)
if fps is None or fps <= 0:
    fps = 30  # fallback
delay = int(1000 / fps)

frame_skip = 1
frame_count = 0

track_history = {}

while True:
    ret, frame = cap.read()

    # End of video safety
    if not ret or frame is None:
        print("End of video or failed frame read.")
        break

    frame_count += 1

    if frame_count % frame_skip != 0:
        continue

    try:
        frame = cv2.resize(frame, (960, 540))
    except Exception as e:
        print("Resize error:", e)
        continue

    try:
        results = model.track(
            frame,
            persist=True,
            conf=0.3,
            iou=0.5,
            tracker="bytetrack.yaml"
        )
    except Exception as e:
        print("Tracking error:", e)
        continue

    # Validate results safely
    if not results or len(results) == 0:
        cv2.imshow("Multi-Object Tracking", frame)
        if cv2.waitKey(delay) == 27:
            break
        continue

    boxes = results[0].boxes

    if boxes is None:
        cv2.imshow("Multi-Object Tracking", frame)
        if cv2.waitKey(delay) == 27:
            break
        continue

    try:
        coords = boxes.xyxy.cpu().numpy()
        classes = boxes.cls.cpu().numpy()
        ids = boxes.id

        if ids is None:
            cv2.imshow("Multi-Object Tracking", frame)
            if cv2.waitKey(delay) == 27:
                break
            continue

        ids = ids.cpu().numpy()

    except Exception as e:
        print("Data extraction error:", e)
        continue

    
    for i in range(min(len(coords), len(classes), len(ids))):
        try:
            box = coords[i]
            cls = int(classes[i])
            track_id = int(ids[i])

            # Only person class
            if cls != 0:
                continue

            x1, y1, x2, y2 = map(int, box)

            
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            cv2.putText(frame,
                        f"ID {track_id}",
                        (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0, 255, 0),
                        2)

            
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

            #
            if track_id not in track_history:
                track_history[track_id] = []

            track_history[track_id].append((cx, cy))

            if len(track_history[track_id]) > 30:
                track_history[track_id].pop(0)

           
            for j in range(1, len(track_history[track_id])):
                cv2.line(frame,
                         track_history[track_id][j - 1],
                         track_history[track_id][j],
                         (0, 255, 0),
                         2)

        except Exception as e:
            print(f"Error in detection loop (index {i}):", e)
            continue

    cv2.imshow("Multi-Object Tracking", frame)

   
    key = cv2.waitKey(delay)
    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 4 persons, 1 umbrella, 251.0ms
Speed: 5.0ms preprocess, 251.0ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 1 umbrella, 249.6ms
Speed: 4.8ms preprocess, 249.6ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 212.5ms
Speed: 4.9ms preprocess, 212.5ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 214.1ms
Speed: 4.1ms preprocess, 214.1ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 207.3ms
Speed: 4.0ms preprocess, 207.3ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 1 umbrella, 197.3ms
Speed: 3.5ms preprocess, 197.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 1 umbrella, 199.2ms
Speed: 5.3ms preprocess, 199.2ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 1 umbrella, 193.8ms
Speed: